# Store Sales — Feature Engineering

This notebook reads the raw data from `data/`, runs preprocessing and feature creation, and saves the results to `artifacts/` for the modeling notebook.

> 🇷🇺 Этот ноутбук читает исходные данные из `data/`, выполняет препроцессинг и создание признаков и сохраняет результат в `artifacts/` для ноутбука моделирования.

In [1]:
import time; _t0 = time.time()

# Base libraries / Базовые библиотеки
import os
import warnings
warnings.filterwarnings('ignore')

# Data handling / Работа с данными
import numpy as np
import pandas as pd

In [2]:
KAGGLE_PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting"
LOCAL_PATH  = "../data"

DATA_PATH = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

train        = pd.read_csv(os.path.join(DATA_PATH, "train.csv"),           parse_dates=["date"])
test         = pd.read_csv(os.path.join(DATA_PATH, "test.csv"),            parse_dates=["date"])
oil          = pd.read_csv(os.path.join(DATA_PATH, "oil.csv"),             parse_dates=["date"])
holidays     = pd.read_csv(os.path.join(DATA_PATH, "holidays_events.csv"), parse_dates=["date"])
stores       = pd.read_csv(os.path.join(DATA_PATH, "stores.csv"))
transactions = pd.read_csv(os.path.join(DATA_PATH, "transactions.csv"),    parse_dates=["date"])

print("DATA_PATH:", DATA_PATH)
print(f"train: {train.shape}, test: {test.shape}")

DATA_PATH: ../data
train: (3000888, 6), test: (28512, 5)


## Feature Engineering

#### Preprocessing

The features below are created here. They split into two groups by purpose.

**Base features — used by the model in notebook 03:**

**Calendar:** `day_of_week`, `month`, `year`, `is_weekend`, `day_of_year`, `week_of_year`, `date_index`.

**Fourier:** `sin_day`, `cos_day`, `sin_week`, `cos_week`.

**Sales history:** `lag_1`–`lag_6`, `lag_7`, `lag_14`, `lag_21`, `lag_28`, `lag_42`, `lag_56`, `lag_364`, `lag_365`, `rolling_mean_7`, `rolling_mean_14`, `rolling_mean_28`, `rolling_mean_364`.

**External factors:** `dcoilwtico`, `dcoilwtico_ma7`, `dcoilwtico_ma28`, `onpromotion`, `onpromotion_ma7`, `onpromotion_ma28`.

**Transactions:** `transactions_lag_16`–`transactions_lag_23`. These lags are safe for the test period because they always point back to train data.

**Holidays and events:** `is_holiday_national`, `day_before_holiday`, `is_black_friday`, `is_cyber_monday`, `is_terremoto`, `is_navidad`, `is_dia_madre`, `is_futbol`, `is_dia_trabajo`, `is_primer_dia`, `is_dia_difuntos`, `work_day`.

**Store and product:** `store_nbr`, `family`, `store_type`, `cluster`.

**Candidate features — created here but used only in experiments 04–09, not in the main model:**

- **Paydays (experiment 05):** `day_of_month`, `is_payday`, `days_since_payday`, `days_to_payday` — wages in Ecuador are paid on the 15th and on the last day of the month.
- **Geography (experiment 05):** `city`, `state`, `is_holiday_local`, `is_holiday_regional` — local/regional holidays matched to each store's location.
- **Dynamic per-series (experiment 07):** `growth_ratio`, `dow_mean_4`, `rolling_std_28`, `zero_share_28`.

The parquet file is a *superset*: each notebook selects the columns it uses via an explicit feature list, so adding candidate columns here never changes the existing models. The candidates were added later, at the experiment stage, and live here so the experiments can read them from one place.

> 🇷🇺 Признаки делятся на две группы по назначению. **Базовые** (идут в модель ноутбука 03): календарные, Fourier, история продаж (лаги и rolling-средние), внешние факторы (нефть, промо), безопасные лаги транзакций, флаги праздников и событий, признаки магазина и товара. **Кандидатские** (создаются здесь, но используются только в экспериментах 04–09, в основную модель не входят): зарплатные дни и география (эксперимент 05); динамические признаки рядов `growth_ratio`, `dow_mean_4`, `rolling_std_28`, `zero_share_28` (эксперимент 07). Parquet — супермножество: каждый ноутбук берёт нужные колонки по явному списку, поэтому добавление кандидатов не меняет существующие модели. Кандидаты добавлены позже, на этапе экспериментов, и лежат здесь, чтобы эксперименты читали их из одного места.

In [3]:
# Add the missing December 25 rows with zero sales / Добавляем пропущенные 25 декабря с нулевыми продажами
# Build the full calendar / Создаём полный календарь
full_idx = pd.MultiIndex.from_product(
    [
        pd.date_range(train["date"].min(), train["date"].max()),
        train["store_nbr"].unique(),
        train["family"].unique()
    ],
    names=["date", "store_nbr", "family"]
)

# Reindex train to add the missing days / Переиндексируем train и добавляем пропущенные дни
train = (
    train
    .set_index(["date", "store_nbr", "family"])
    .reindex(full_idx)
    .reset_index()
)

# Fill the new rows / Заполняем новые строки
train["sales"] = train["sales"].fillna(0)
train["onpromotion"] = train["onpromotion"].fillna(0)

In [4]:
# Build the full list of train dates / Создаём полный список дат train
train_dates = pd.date_range(
    start=train["date"].min(),   # first date in train / первая дата в train
    end=train["date"].max(),     # last date in train / последняя дата в train
    freq="D"                     # step = 1 day / шаг = 1 день
)

# Dates present in oil / Даты, которые есть в oil
oil_dates = pd.Index(oil["date"].unique())

# Train dates missing from oil / Даты train, отсутствующие в oil
missing_oil_dates = train_dates.difference(oil_dates)

print("Days in train period:", len(train_dates))
print("Dates in oil:", oil["date"].nunique())
print("Train dates missing from oil:", len(missing_oil_dates))

# First missing dates / Первые пропущенные даты
missing_oil_dates[:20]

Days in train period: 1688
Dates in oil: 1218
Train dates missing from oil: 482


DatetimeIndex(['2013-01-05', '2013-01-06', '2013-01-12', '2013-01-13',
               '2013-01-19', '2013-01-20', '2013-01-26', '2013-01-27',
               '2013-02-02', '2013-02-03', '2013-02-09', '2013-02-10',
               '2013-02-16', '2013-02-17', '2013-02-23', '2013-02-24',
               '2013-03-02', '2013-03-03', '2013-03-09', '2013-03-10'],
              dtype='datetime64[ns]', freq=None)

In [5]:
# Rebuild the daily oil calendar / Восстанавливаем ежедневный календарь oil
train_start = train["date"].min()
test_end = test["date"].max()

oil = (
    oil
    .set_index("date")
    .reindex(pd.date_range(train_start, test_end, freq="D"))
    .rename_axis("date")
    .sort_index()
    .reset_index()
)

# Fill gaps with linear interpolation / Заполняем пропуски линейной интерполяцией
oil["dcoilwtico"] = oil["dcoilwtico"].interpolate(
    method="linear",
    limit_direction="both"
)

# Round the oil price to 2 decimals / Округляем цену нефти до 2 знаков
oil["dcoilwtico"] = oil["dcoilwtico"].round(2)

print("Oil date range after rebuild:")
print(oil["date"].min())
print(oil["date"].max())

print("Missing values:")
print(oil.isnull().sum())

Oil date range after rebuild:
2013-01-01 00:00:00
2017-08-31 00:00:00
Missing values:
date          0
dcoilwtico    0
dtype: int64


In [6]:
# Rebuild the daily transactions series per store / Восстанавливаем ежедневный ряд transactions по каждому магазину
# Total sales per store and date / Продажи по магазину и дате
store_sales = (
    train
    .groupby(["date", "store_nbr"])["sales"]
    .sum()
    .reset_index()
)

# Full calendar: all dates x all stores / Полный календарь: все даты x все магазины
full_idx = pd.MultiIndex.from_product(
    [
        pd.date_range(train["date"].min(), train["date"].max(), freq="D"),
        train["store_nbr"].unique()
    ],
    names=["date", "store_nbr"]
)

# Add the missing date x store pairs / Добавляем пропущенные пары дата x магазин
store_sales = (
    store_sales
    .set_index(["date", "store_nbr"])
    .reindex(full_idx)
    .reset_index()
)

# Merge transactions with sales / Объединяем транзакции с продажами
transactions = (
    transactions
    .merge(store_sales, on=["date", "store_nbr"], how="outer")
    .sort_values(["store_nbr", "date"])
)

# No sales -> transactions = 0 / Нет продаж -> транзакции = 0
transactions.loc[transactions["sales"] == 0, "transactions"] = 0

# Drop the helper sales column / Удаляем временный столбец sales
transactions = transactions.drop(columns=["sales"])

# Interpolate the remaining gaps per store / Интерполируем оставшиеся пропуски по каждому магазину
transactions["transactions"] = (
    transactions
    .groupby("store_nbr")["transactions"]
    .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

print("Missing values after processing:")
print(transactions.isna().sum())

Missing values after processing:
date            0
store_nbr       0
transactions    0
dtype: int64


#### Creating the features

In [7]:
# Concatenate train and test so features are computed the same way / Объединяем train и test, чтобы считать признаки одинаково
df = pd.concat([train, test], ignore_index=True, sort=False)
df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

# Calendar features / Календарные признаки
df["day_of_week"]  = df["date"].dt.dayofweek
df["month"]        = df["date"].dt.month
df["year"]         = df["date"].dt.year
df["is_weekend"]   = (df["date"].dt.dayofweek >= 5).astype(int)
df["day_of_year"]  = df["date"].dt.dayofyear
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["date_index"]   = (df["date"] - df["date"].min()).dt.days

# Fourier features for smooth yearly and weekly seasonality / Fourier-признаки плавной годовой и недельной сезонности
df["sin_day"]  = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["cos_day"]  = np.cos(2 * np.pi * df["day_of_year"] / 365)
df["sin_week"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["cos_week"] = np.cos(2 * np.pi * df["day_of_week"] / 7)


In [8]:
# Lag features / Лаговые признаки
# Lags 1-6: short-term dynamics / Лаги 1-6: краткосрочная динамика
# Lags 7/14/21/28/42/56: weekly and monthly cycles / Лаги 7/14/21/28/42/56: недельные и месячные циклы
# Lags 364/365: sales about one year ago / Лаги 364/365: продажи примерно год назад
for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]:
    df[f"lag_{lag}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .shift(lag)
    )

# Rolling mean over past N days, current day excluded / Rolling mean: среднее прошлых N дней, без текущего дня
for window in [7, 14, 28]:
    df[f"rolling_mean_{window}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

# Yearly rolling mean / Годовое скользящее среднее
df["rolling_mean_364"] = (
    df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(364, min_periods=30).mean())
)


In [9]:
# Dynamic per-series features (candidates for experiment 07), leak-free via shift(1):
# growth_ratio — level vs a year ago; dow_mean_4 — mean of last 4 same-weekday sales;
# rolling_std_28 — volatility; zero_share_28 — share of recent zero-sales days.
# (days_since_last_sale is skipped on purpose: it degenerates in recursive forecasting — see nb 07.)
# Динамические фичи рядов (кандидаты для эксп. 07), без утечки через shift(1):
# growth_ratio — уровень против года назад; dow_mean_4 — среднее 4 последних тех же дней недели;
# rolling_std_28 — волатильность; zero_share_28 — доля недавних нулевых дней.
# (days_since_last_sale намеренно пропущен: вырождается в рекурсивном прогнозе — см. nb 07.)
df["growth_ratio"] = (
    (df["rolling_mean_28"] / df["rolling_mean_364"])
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
)
df["dow_mean_4"] = df[["lag_7", "lag_14", "lag_21", "lag_28"]].mean(axis=1)
df["rolling_std_28"] = (
    df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(28, min_periods=2).std())
)
df["_zero"] = (df["sales"] == 0).astype(float)
df["zero_share_28"] = (
    df.groupby(["store_nbr", "family"])["_zero"]
    .transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
)
df = df.drop(columns=["_zero"])

In [10]:
# Add the oil price / Добавляем цену нефти
df = df.merge(oil[["date", "dcoilwtico"]], on="date", how="left")

# Add store type and cluster / Добавляем тип и кластер магазина
df = df.merge(
    stores[["store_nbr", "type", "cluster"]],
    on="store_nbr", how="left"
).rename(columns={"type": "store_type"})

# Transaction lags 16-23 days back / Лаги транзакций 16-23 дня назад
# For test these lags always point to the train period / Для test эти лаги всегда указывают на train-период
# Example: 2017-08-16 minus 16 days = 2017-07-31 / Например: 2017-08-16 минус 16 дней = 2017-07-31
df = df.merge(
    transactions[["date", "store_nbr", "transactions"]],
    on=["date", "store_nbr"], how="left"
)
for lag in range(16, 24):
    df[f"transactions_lag_{lag}"] = df.groupby("store_nbr")["transactions"].shift(lag)
df = df.drop(columns=["transactions"])

# Moving averages of oil and promotions / Скользящие средние нефти и промо
for col in ["dcoilwtico", "onpromotion"]:
    for w in [7, 28]:
        df[f"{col}_ma{w}"] = (
            df.groupby(["store_nbr", "family"])[col]
            .transform(lambda x: x.rolling(w, min_periods=1).mean())
        )


In [11]:
# National holidays without transferred dates / Национальные праздники без перенесённых дат
national_hol = holidays[
    (holidays["locale"] == "National") &
    (holidays["transferred"] == False)
]["date"].unique()

df["is_holiday_national"] = df["date"].isin(national_hol).astype(int)
df["day_before_holiday"]  = (df["date"] + pd.Timedelta(days=1)).isin(national_hol).astype(int)

# Commercial events / Коммерческие события
df["is_black_friday"] = df["date"].isin(
    holidays[holidays["description"] == "Black Friday"]["date"].unique()
).astype(int)
df["is_cyber_monday"] = df["date"].isin(
    holidays[holidays["description"] == "Cyber Monday"]["date"].unique()
).astype(int)

# Manabi earthquake period / Период землетрясения Manabi
df["is_terremoto"] = df["date"].isin(
    holidays[holidays["description"].str.contains("Terremoto", na=False)]["date"].unique()
).astype(int)

# Individual national holidays / Отдельные национальные праздники
_hol = holidays[(holidays["locale"] == "National") & (holidays["transferred"] == False)]
df["is_navidad"]       = df["date"].isin(_hol[_hol["description"].str.contains("Navidad",      na=False)]["date"].unique()).astype(int)
df["is_dia_madre"]     = df["date"].isin(_hol[_hol["description"].str.contains("Madre",        na=False)]["date"].unique()).astype(int)
df["is_futbol"]        = df["date"].isin(_hol[_hol["description"].str.contains("futbol|F\xfatbol|Mundial", na=False)]["date"].unique()).astype(int)
df["is_dia_trabajo"]   = df["date"].isin(_hol[_hol["description"].str.contains("Trabajo",      na=False)]["date"].unique()).astype(int)
df["is_primer_dia"]    = df["date"].isin(_hol[_hol["description"].str.contains("Primer",       na=False)]["date"].unique()).astype(int)
df["is_dia_difuntos"]  = df["date"].isin(_hol[_hol["description"].str.contains("Difuntos",     na=False)]["date"].unique()).astype(int)

# Working days marked in holidays / Рабочие дни, отмеченные в holidays
work_day_dates = holidays[
    (holidays["type"] == "Work Day") |
    (holidays["description"].str.contains("trabajo|Work", case=False, na=False) & (holidays["locale"] == "National"))
]["date"].unique()
df["work_day"] = df["date"].isin(work_day_dates).astype(int)

In [12]:
# Paydays and local geography (feature candidates for experiment 05)
# Зарплатные дни и локальная география (фичи-кандидаты для эксперимента 05)

# Wages in Ecuador are paid on the 15th and on the last day of the month (from the competition data description),
# so demand spikes right after these dates. All four features are pure date arithmetic — leak-free by construction.
# Зарплаты в Эквадоре платят 15-го и в последний день месяца (из описания данных соревнования),
# поэтому спрос растёт сразу после этих дат. Все четыре фичи — чистая арифметика дат, без утечек по построению.
dom = df["date"].dt.day
dim = df["date"].dt.days_in_month
df["day_of_month"]      = dom
df["is_payday"]         = ((dom == 15) | (dom == dim)).astype(int)
# Days since the most recent payday / until the next one (0 on the payday itself)
# Дней с последней зарплаты / до следующей (в день зарплаты = 0)
df["days_since_payday"] = np.where(dom == dim, 0, np.where(dom >= 15, dom - 15, dom))
df["days_to_payday"]    = np.where(dom <= 15, 15 - dom, dim - dom)

# Store city/state + local and regional holidays matched to the store's location
# (same logic as is_holiday_national: transferred dates excluded)
# Город/штат магазина + локальные и региональные праздники, сопоставленные с локацией магазина
# (та же логика, что в is_holiday_national: перенесённые даты исключаются)
df = df.merge(stores[["store_nbr", "city", "state"]], on="store_nbr", how="left")

local_hol    = holidays[(holidays["locale"] == "Local")    & (holidays["transferred"] == False)]
regional_hol = holidays[(holidays["locale"] == "Regional") & (holidays["transferred"] == False)]
local_set    = set(zip(local_hol["date"],    local_hol["locale_name"]))
regional_set = set(zip(regional_hol["date"], regional_hol["locale_name"]))

df["is_holiday_local"]    = np.fromiter(((d, c) in local_set    for d, c in zip(df["date"], df["city"])),  dtype=int, count=len(df))
df["is_holiday_regional"] = np.fromiter(((d, s) in regional_set for d, s in zip(df["date"], df["state"])), dtype=int, count=len(df))

print("payday dates:", int(df.loc[df["is_payday"] == 1, "date"].nunique()), "of", df["date"].nunique())
print("local holiday rows:", int(df["is_holiday_local"].sum()),
      "| regional holiday rows:", int(df["is_holiday_regional"].sum()))

payday dates: 112 of 1704
local holiday rows: 11946 | regional holiday rows: 1023


In [13]:
# Split the data back into train and test / Разделяем данные обратно на train и test
train_fe = df[df["sales"].notna()].copy()
test_fe  = df[df["sales"].isna()].copy()

train_fe = train_fe.drop(columns=["id", "day_name"], errors="ignore")
test_fe  = test_fe.drop(columns=["id", "day_name"], errors="ignore")

print("train_fe shape:", train_fe.shape)
print("test_fe shape: ", test_fe.shape)
print()
print("Features:", train_fe.columns.tolist())
print()
print("Missing values in train_fe:")
print(train_fe.isnull().sum()[train_fe.isnull().sum() > 0])


train_fe shape: (3008016, 73)
test_fe shape:  (28512, 73)

Features: ['date', 'store_nbr', 'family', 'sales', 'onpromotion', 'day_of_week', 'month', 'year', 'is_weekend', 'day_of_year', 'week_of_year', 'date_index', 'sin_day', 'cos_day', 'sin_week', 'cos_week', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_42', 'lag_56', 'lag_364', 'lag_365', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_364', 'growth_ratio', 'dow_mean_4', 'rolling_std_28', 'zero_share_28', 'dcoilwtico', 'store_type', 'cluster', 'transactions_lag_16', 'transactions_lag_17', 'transactions_lag_18', 'transactions_lag_19', 'transactions_lag_20', 'transactions_lag_21', 'transactions_lag_22', 'transactions_lag_23', 'dcoilwtico_ma7', 'dcoilwtico_ma28', 'onpromotion_ma7', 'onpromotion_ma28', 'is_holiday_national', 'day_before_holiday', 'is_black_friday', 'is_cyber_monday', 'is_terremoto', 'is_navidad', 'is_dia_madre', 'is_futbol', 'is_dia_trabajo', 'is

In [14]:
# Save artifacts for notebook 03 / Сохраняем артефакты для ноутбука 03
# df: train+test with all features; train: after calendar rebuild / df: train+test со всеми признаками; train: после восстановления календаря
os.makedirs("../artifacts", exist_ok=True)

df.to_parquet("../artifacts/features.parquet", index=False)
train.to_parquet("../artifacts/train_preprocessed.parquet", index=False)

print("Saved:")
print("  ../artifacts/features.parquet          ", df.shape)
print("  ../artifacts/train_preprocessed.parquet", train.shape)
print(f"Runtime: {(time.time() - _t0) / 60:.1f} min")

Saved:
  ../artifacts/features.parquet           (3036528, 74)
  ../artifacts/train_preprocessed.parquet (3008016, 6)
Runtime: 0.5 min
